# Wind & Solar Capacity Predictor

Target => val_geracaoverificada

In [ ]:
import gc
import pandas as pd
import numpy as np
from os import listdir
from pathlib import Path
from sklearn.neighbors import BallTree
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, train_test_split, TimeSeriesSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder


Agregando os dados extraidos da ONS:

In [ ]:
df = pd.concat(
    [pd.read_parquet(f"./database/{file}") for file in listdir("./database") if file.endswith('.parquet')],
    ignore_index=True
)

Ajustando a tipagem das colunas do DF de acordo com a documentação:

In [ ]:
float_columns = [
    "val_latitudesecoletora",
    "val_longitudesecoletora",
    "val_latitudepontoconexao",
    "val_longitudepontoconexao",
    "val_geracaoverificada",
    "val_capacidadeinstalada",
    "val_fatorcapacidade"
]

df[float_columns] = df[float_columns].apply(pd.to_numeric, errors="coerce")

df["din_instante"] = pd.to_datetime(df["din_instante"])

df = df.drop(['nom_subsistema', 'cod_pontoconexao', 'ceg', 'nom_estado', 'val_fatorcapacidade', 'val_geracaoprogramada'],axis=1) # fator de capacidade é derivado da capacidade gerada e geracao programada é literalmente a previsão da gereção

In [ ]:
print(df.sort_values(by="din_instante").iloc[0]) # Encontra o registro mais antigo
ano_alvo = 2024 #recorte por ano por limitação de hardware

Lê os dados metereológicos extraídos do [INMET](https://portal.inmet.gov.br/dadoshistoricos) a partir de 2009 até 2026:

In [ ]:
def _num(valor):
    """Converte '1159,54' -> 1159.54; qualquer valor inválido vira NaN."""
    return pd.to_numeric(str(valor).replace(",", "."), errors="coerce")

def ler_inmet(caminho):
    # 1) Metadados (8 primeiras linhas)
    meta = {}
    with open(caminho, encoding="latin-1") as f:
        for _ in range(8):
            chave, _, valor = f.readline().strip().partition(";")
            meta[chave.rstrip(":")] = valor.strip()

    # 2) Dados
    df = pd.read_csv(
        caminho,
        sep=";",
        skiprows=8,
        decimal=",",
        encoding="latin-1",
        na_values=["-9999"],
        index_col=False,
    )
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

    df["REGIAO"] = meta.get("REGIÃO")
    df["UF"] = meta.get("UF")
    df["ESTACAO"] = meta.get("ESTAÇÃO")
    df["CODIGO_WMO"] = meta.get("CODIGO (WMO)")
    df["LATITUDE"] = _num(meta.get("LATITUDE"))
    df["LONGITUDE"] = _num(meta.get("LONGITUDE"))
    df["ALTITUDE"] = _num(meta.get("ALTITUDE"))
    df["DATA_FUNDACAO"] = pd.to_datetime(
        meta.get("DATA DE FUNDAÇÃO (YYYY-MM-DD)"), errors="coerce"
    )

    return df

pasta = Path(f"./database/INMET/{ano_alvo}")

df_weather = pd.concat(
    [ler_inmet(pasta / f) for f in listdir(pasta) if f.lower().endswith(".csv")],
    ignore_index=True,
)

In [ ]:
def padronizar_colunas(df: pd.DataFrame) -> pd.DataFrame:
    """Normaliza nomes de colunas que mudaram ao longo dos anos no INMET.

    `rename` ignora chaves ausentes, então não depende de cortes de ano.
    """
    return df.rename(columns={
        # formato até 2018
        "DATA (YYYY-MM-DD)": "DATA",
        "HORA (UTC)": "HORA",
        # formato a partir de 2019
        "Data": "DATA",
        "Hora UTC": "HORA",
        # radiação (KJ vs Kj)
        "RADIACAO GLOBAL (KJ/m²)": "RADIACAO GLOBAL (Kj/m²)",
    })


FLOAT_COLUMNS = [
    "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)",
    "PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",
    "PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)",
    "PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)",
    "RADIACAO GLOBAL (Kj/m²)",
    "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",
    "TEMPERATURA DO PONTO DE ORVALHO (°C)",
    "TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)",
    "TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)",
    "TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)",
    "TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)",
    "UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)",
    "UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)",
    "UMIDADE RELATIVA DO AR, HORARIA (%)",
    "VENTO, DIREÇÃO HORARIA (gr) (° (gr))",
    "VENTO, RAJADA MAXIMA (m/s)",
    "VENTO, VELOCIDADE HORARIA (m/s)",
    "LATITUDE",
    "LONGITUDE",
    "ALTITUDE",
]
CATEGORY_COLUMNS = ["REGIAO", "UF", "ESTACAO", "CODIGO_WMO"]

df_weather = padronizar_colunas(df_weather)

df_weather[FLOAT_COLUMNS] = df_weather[FLOAT_COLUMNS].apply(pd.to_numeric, errors="coerce")

hora = (
    df_weather["HORA"]
    .astype(str)
    .str.replace(" UTC", "", regex=False)
    .str.replace(":", "", regex=False)
    .str.zfill(4)
)
data = df_weather["DATA"].astype(str).str.replace("/", "-", regex=False)  # aceita 2019/01/01 e 2019-01-01

df_weather["DATETIME"] = pd.to_datetime(
    data + " " + hora,
    format="%Y-%m-%d %H%M",
    errors="coerce",
    utc=True,
)
df_weather["DATA_FUNDACAO"] = pd.to_datetime(df_weather["DATA_FUNDACAO"], errors="coerce")

df_weather = df_weather.drop(columns=["DATA", "HORA"])
df_weather[CATEGORY_COLUMNS] = df_weather[CATEGORY_COLUMNS].astype("category")

Join dos Datasets:

In [ ]:
# O INMET aqui é só de 20XX, então filtra o df para o mesmo ano
df_anual = df[df["din_instante"].dt.year == ano_alvo].copy()

# Garbage collector
del df
gc.collect()

In [ ]:
# ------------------------------------------------------------
# Prepara o weather
# ------------------------------------------------------------
weather = df_weather.copy()

weather["CODIGO_WMO"] = weather["CODIGO_WMO"].astype(str)
weather = weather.drop_duplicates(["CODIGO_WMO", "DATETIME"])

# Garbage collector
del df_weather
gc.collect()

In [ ]:
# ------------------------------------------------------------
# 1) Usina -> estação mais próxima
# ------------------------------------------------------------
plantas = (
    df_anual[["id_ons", "val_latitudesecoletora", "val_longitudesecoletora"]]
    .dropna()
    .drop_duplicates("id_ons")
    .reset_index(drop=True)
)

estacoes = (
    weather[["CODIGO_WMO", "LATITUDE", "LONGITUDE"]]
    .dropna()
    .drop_duplicates("CODIGO_WMO")
    .reset_index(drop=True)
)

tree = BallTree(
    np.radians(estacoes[["LATITUDE", "LONGITUDE"]].to_numpy()),
    metric="haversine",
)
dist, idx = tree.query(
    np.radians(plantas[["val_latitudesecoletora", "val_longitudesecoletora"]].to_numpy()),
    k=1,
)

plantas["CODIGO_WMO"] = estacoes["CODIGO_WMO"].to_numpy()[idx[:, 0]]
plantas["dist_estacao_km"] = dist[:, 0] * 6371  # raio da Terra em km

# ------------------------------------------------------------
# 2) Alinha o tempo (ONS = horário de Brasília, INMET = UTC)
# ------------------------------------------------------------
df_anual["DATETIME"] = (
    (df_anual["din_instante"] + pd.Timedelta(hours=3))
    .dt.tz_localize("UTC")
    .astype("datetime64[ns, UTC]")
)
weather["DATETIME"] = weather["DATETIME"].astype("datetime64[ns, UTC]")

# ------------------------------------------------------------
# 3) Join
# ------------------------------------------------------------
df_anual = df_anual.merge(
    plantas[["id_ons", "CODIGO_WMO", "dist_estacao_km"]],
    on="id_ons",
    how="left",
)

weather = weather.rename(columns={
    "LATITUDE": "LAT_ESTACAO",
    "LONGITUDE": "LON_ESTACAO",
    "ALTITUDE": "ALT_ESTACAO",
})

df_final = df_anual.merge(
    weather,
    on=["CODIGO_WMO", "DATETIME"],
    how="left",
    validate="many_to_one",
)

In [ ]:
# Garbage Collector
del df_anual
del weather
gc.collect()

In [ ]:
df_final = df_final.drop(['val_latitudesecoletora', 'val_longitudesecoletora', 'LAT_ESTACAO', 'LON_ESTACAO', 'id_subsistema', 'id_estado', 'CODIGO_WMO'],axis=1)

In [ ]:
print(df_final.iloc[1000])
print((df_final['nom_tipousina'] != 'Eólica').mean())  # fração de registros não eólicos

Consulta SQL 1)

Consulta SQL 2)

Consulta SQL 3)

Consulta SQL 4)

Consulta SQL 5)

# Modelagem:

In [ ]:
df_modelo = (
    df_final[df_final['nom_tipousina'] != 'Eólica']
    .dropna(subset=['val_geracaoverificada'])
    .sort_values('din_instante')          # ordem cronológica p/ TimeSeriesSplit
    .reset_index(drop=True)
)

# Features temporais úteis, extraídas antes de descartar as datas
df_modelo['hora'] = df_modelo['din_instante'].dt.hour
df_modelo['mes'] = df_modelo['din_instante'].dt.month
df_modelo['dia_semana'] = df_modelo['din_instante'].dt.dayofweek
df_modelo['idade_estacao_anos'] = (
    (df_modelo['din_instante'].dt.tz_localize(None)
     - df_modelo['DATA_FUNDACAO'].dt.tz_localize(None)).dt.days / 365.25
)

y = df_modelo['val_geracaoverificada']

X = df_modelo.drop(columns=[
    'val_geracaoverificada',
    'din_instante', 'DATETIME', 'DATA_FUNDACAO',
    'nom_tipousina',         # constante após o filtro
    'id_ons',                # identificador
])

# Corte temporal: últimos 10% como hold-out
corte = int(len(X) * 0.9)
X_train, X_test = X.iloc[:corte], X.iloc[corte:]
y_train, y_test = y.iloc[:corte], y.iloc[corte:]

## Baseline:

## Modelos Obrigatórios: Random Forest e SVR

### Random Forest:

In [ ]:
cat_cols = X_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocess = ColumnTransformer([
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
    ('num', 'passthrough', num_cols),
])

pipeline = Pipeline([
    ('prep', preprocess),
    ('rf', RandomForestRegressor(random_state=42, n_jobs=-1)),
])

param_grid = {
    'rf__n_estimators':      [200],
    'rf__max_features':      ['sqrt'],
    'rf__bootstrap':         [True],
    'rf__criterion':         ['squared_error'],
    'rf__max_depth':         [20],
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=TimeSeriesSplit(n_splits=5),
    scoring='neg_root_mean_squared_error',
    n_jobs=3,
    verbose=1
)

In [ ]:
grid_search.fit(X_train, y_train)
print(f"Melhores parâmetros: {grid_search.best_params_}")

In [ ]:
# ------------------------------------------------------------
# Resultado do CV
# ------------------------------------------------------------
print(f"Melhores parâmetros: {grid_search.best_params_}")
print(f"RMSE médio no CV (treino): {-grid_search.best_score_:.4f}\n")

# ------------------------------------------------------------
# Predição no hold-out
# ------------------------------------------------------------
best_model = grid_search.best_estimator_
y_pred = np.clip(best_model.predict(X_test), 0, None)   # geração não é negativa
y_true = y_test.to_numpy()
cap = X_test['val_capacidadeinstalada'].to_numpy()

# ------------------------------------------------------------
# Métricas base
# ------------------------------------------------------------
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

erro_abs = np.abs(y_true - y_pred)
mask_cap = cap > 0
erro_pct_cap = np.where(mask_cap, erro_abs / np.where(mask_cap, cap, 1) * 100, np.nan)

nmae = np.nanmean(erro_pct_cap)
wape = erro_abs.sum() / y_true.sum() * 100

# Baseline ingênuo: prever sempre a média do treino
y_base = np.full_like(y_true, y_train.mean(), dtype=float)
rmse_base = np.sqrt(mean_squared_error(y_true, y_base))
mae_base = mean_absolute_error(y_true, y_base)

# ------------------------------------------------------------
# Métricas intuitivas
# ------------------------------------------------------------
# 1) Confiabilidade geral: quanto do volume gerado o modelo acerta
confiabilidade = max(0.0, 100 - wape)

# 2) Diferença percentual total: previsto vs real (soma de tudo)
total_real = y_true.sum()
total_prev = y_pred.sum()
dif_total_pct = (total_prev - total_real) / total_real * 100

# 3) Diferença percentual média por amostra (só onde o real > 0, evita divisão por zero)
mask_real = y_true > 0
dif_pct_amostra = np.abs(y_pred[mask_real] - y_true[mask_real]) / y_true[mask_real] * 100
dif_pct_mediana = np.median(dif_pct_amostra)

# 4) Taxa de acerto por faixa de erro (em % da capacidade instalada)
def taxa_acerto(tol_pct):
    return np.nanmean(erro_pct_cap <= tol_pct) * 100

# 5) Classificação textual da confiabilidade
def classificar(c):
    if c >= 90: return "EXCELENTE"
    if c >= 80: return "MUITO BOA"
    if c >= 70: return "BOA"
    if c >= 60: return "REGULAR"
    return "BAIXA"

# 6) Tendência: o modelo superestima ou subestima?
vies = "superestima" if dif_total_pct > 0 else "subestima"

# ------------------------------------------------------------
# Relatório
# ------------------------------------------------------------
print("=" * 60)
print(f"  RELATÓRIO DE DESEMPENHO DO MODELO — {len(y_true):,} amostras de teste")
print("=" * 60)

print(f"\n>> CONFIABILIDADE GERAL: {confiabilidade:.1f}%  [{classificar(confiabilidade)}]")
print(f"   O modelo acerta ~{confiabilidade:.0f}% do volume de energia gerada.")

print("\n>> PREVISTO vs REAL")
print(f"   Geração real total:     {total_real:>14,.0f} MW")
print(f"   Geração prevista total: {total_prev:>14,.0f} MW")
print(f"   Diferença total:        {dif_total_pct:>+13.2f}%  (o modelo {vies} levemente)")
print(f"   Erro típico por amostra: {dif_pct_mediana:.1f}% (mediana)")

print("\n>> ERRO EM MW")
print(f"   Em média, a previsão erra por {mae:.1f} MW (baseline ingênuo: {mae_base:.1f} MW)")
print(f"   Erro médio equivale a {nmae:.1f}% da capacidade instalada")

print("\n>> TAXA DE ACERTO (previsão dentro de X% da capacidade instalada)")
for tol in (0, 0.1, 1, 5, 10, 15, 20):
    t = taxa_acerto(tol)
    barra = "█" * int(t / 5) + "░" * (20 - int(t / 5))
    print(f"   até {tol:>3}%:  {barra}  {t:5.1f}%")

print("\n>> COMPARAÇÃO COM A MÉDIA")
print(f"   Melhora no erro (RMSE): {(1 - rmse / rmse_base) * 100:.1f}% melhor que o baseline")
print(f"   R² = {r2:.3f}  → o modelo explica {r2 * 100:.1f}% da variação da geração")

print("\n" + "=" * 60)

# ------------------------------------------------------------
# Salva as predições
# ------------------------------------------------------------
np.savetxt('conjunto_teste.predict', y_pred, fmt='%.3f')

### SVR:

## Gradient Boosting:

## TabPFN: